# Notebook 01 — Data Loading, Preprocessing & Exploration

**Dataset:** 10x Genomics Xenium In Situ — Human Breast Cancer (FFPE), Section 1 Top  
**Goal:** Load all data files, apply quality filtering, and visualise the spatial distribution of transcripts.

In [ ]:
import sys
sys.path.insert(0, "..")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tifffile
from pathlib import Path

DATA_DIR = Path("../data/Human_Breast_Biomarkers_S1_Top_outs")
RESULTS_DIR = Path("../results")
RESULTS_DIR.mkdir(exist_ok=True)

print("Data directory contents:")
for f in sorted(DATA_DIR.iterdir()):
    print(f"  {f.name}")

## 1. Load Transcripts

In [ ]:
transcripts = pd.read_parquet(DATA_DIR / "transcripts.parquet")
print(f"Total transcripts: {len(transcripts):,}")
print(f"Columns: {transcripts.columns.tolist()}")
transcripts.head()

## 2. Quality Filtering

Keep only real gene transcripts (not control probes) with Q-score ≥ 20.

In [ ]:
# Filter: real genes only + Q >= 20
transcripts_genes = transcripts[transcripts["is_gene"] == True].copy()
print(f"Gene transcripts (before Q filter): {len(transcripts_genes):,}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(transcripts_genes["qv"], bins=50, color="steelblue", edgecolor="none")
axes[0].axvline(20, color="red", linestyle="--", label="Q20 threshold")
axes[0].set_xlabel("Q-score")
axes[0].set_ylabel("Count")
axes[0].set_title("Q-score Distribution")
axes[0].legend()

transcripts_filtered = transcripts_genes[transcripts_genes["qv"] >= 20].copy()

labels = ["All genes", "Q ≥ 20"]
counts = [len(transcripts_genes), len(transcripts_filtered)]
axes[1].bar(labels, counts, color=["steelblue", "seagreen"])
axes[1].set_ylabel("Transcript count")
axes[1].set_title("Before vs. After Quality Filtering")
for i, v in enumerate(counts):
    axes[1].text(i, v + 50000, f"{v:,}", ha="center", fontsize=10)

plt.tight_layout()
plt.savefig(RESULTS_DIR / "qscore_filtering.png", dpi=150)
plt.show()

pct = 100 * len(transcripts_filtered) / len(transcripts_genes)
print(f"Retained {len(transcripts_filtered):,} transcripts ({pct:.1f}%) after Q ≥ 20 filter.")

## 3. Gene Panel Overview

In [ ]:
gene_counts = transcripts_filtered["feature_name"].value_counts()
print(f"Genes in panel: {len(gene_counts)}")
print(f"\nTop 10 most detected:")
print(gene_counts.head(10).to_string())

fig, ax = plt.subplots(figsize=(12, 4))
gene_counts.head(30).plot(kind="bar", ax=ax, color="steelblue", edgecolor="none")
ax.set_xlabel("Gene")
ax.set_ylabel("Transcript count")
ax.set_title("Top 30 Most Detected Genes (Q ≥ 20)")
plt.tight_layout()
plt.savefig(RESULTS_DIR / "top_genes.png", dpi=150)
plt.show()

## 4. Spatial Distribution of Transcripts

In [ ]:
# Subsample for plotting speed
sample = transcripts_filtered.sample(n=500_000, random_state=42)

fig, ax = plt.subplots(figsize=(9, 9))
ax.scatter(sample["x_location"], sample["y_location"], s=0.1, alpha=0.1, color="steelblue")
ax.set_xlabel("X (µm)")
ax.set_ylabel("Y (µm)")
ax.set_title("Spatial Distribution of Transcripts (500k sample, Q ≥ 20)")
ax.set_aspect("equal")
plt.tight_layout()
plt.savefig(RESULTS_DIR / "transcript_spatial_distribution.png", dpi=150)
plt.show()

## 5. Marker Gene Spatial Patterns

In [ ]:
# Pick genes likely in the 280-gene breast cancer panel
available_genes = gene_counts.index.tolist()
candidates = ["EPCAM", "CD68", "CD3E", "KRT17", "KRT8", "CD45", "VIM", "ACTA2"]
marker_genes = [g for g in candidates if g in available_genes][:4]
print(f"Plotting markers: {marker_genes}")

fig, axes = plt.subplots(2, 2, figsize=(14, 14))
axes = axes.flatten()
colors = ["tomato", "steelblue", "seagreen", "darkorange"]

for ax, gene, color in zip(axes, marker_genes, colors):
    subset = transcripts_filtered[transcripts_filtered["feature_name"] == gene]
    ax.scatter(subset["x_location"], subset["y_location"], s=0.3, alpha=0.4, color=color)
    ax.set_title(f"{gene} ({len(subset):,} transcripts)", fontsize=13)
    ax.set_aspect("equal")
    ax.axis("off")

fig.suptitle("Marker Gene Spatial Patterns", fontsize=15, y=1.01)
plt.tight_layout()
plt.savefig(RESULTS_DIR / "marker_genes_spatial.png", dpi=150, bbox_inches="tight")
plt.show()

## 6. Load 10x Reference Data

In [ ]:
nucleus_boundaries = pd.read_parquet(DATA_DIR / "nucleus_boundaries.parquet")
cell_boundaries = pd.read_parquet(DATA_DIR / "cell_boundaries.parquet")
cells = pd.read_parquet(DATA_DIR / "cells.parquet")

print(f"Cells (10x segmentation): {len(cells):,}")
print(f"Nucleus boundary vertices: {len(nucleus_boundaries):,}")
print(f"Cell boundary vertices: {len(cell_boundaries):,}")
print(f"\nSegmentation method: {cells['segmentation_method'].iloc[0]}")
print(f"\nCell area stats (µm²):")
print(cells["cell_area"].describe().round(2).to_string())

In [ ]:
# Transcripts already assigned by 10x
assigned_10x = transcripts_filtered[transcripts_filtered["cell_id"] != "UNASSIGNED"]
unassigned_10x = transcripts_filtered[transcripts_filtered["cell_id"] == "UNASSIGNED"]

print(f"10x assigned transcripts:   {len(assigned_10x):,} ({100*len(assigned_10x)/len(transcripts_filtered):.1f}%)")
print(f"10x unassigned transcripts: {len(unassigned_10x):,} ({100*len(unassigned_10x)/len(transcripts_filtered):.1f}%)")

per_cell_10x = assigned_10x.groupby("cell_id").size()
print(f"\nTranscripts per cell (10x) — mean: {per_cell_10x.mean():.1f}, median: {per_cell_10x.median():.1f}")

## 7. Load DAPI Image

In [ ]:
dapi = tifffile.imread(DATA_DIR / "morphology_focus" / "ch0000_dapi.ome.tif")
print(f"DAPI image shape: {dapi.shape}, dtype: {dapi.dtype}")

# If multi-page, take first z-slice
if dapi.ndim == 3:
    dapi = dapi[0]
    print(f"Using first z-slice: {dapi.shape}")

dapi_norm = (dapi.astype(float) - dapi.min()) / (dapi.max() - dapi.min())

fig, ax = plt.subplots(figsize=(9, 9))
ax.imshow(dapi_norm, cmap="gray", origin="lower")
ax.set_title("DAPI Nuclear Staining (morphology_focus)")
ax.axis("off")
plt.tight_layout()
plt.savefig(RESULTS_DIR / "dapi_image.png", dpi=150)
plt.show()

## 8. Save Filtered Transcripts

In [ ]:
transcripts_filtered.to_parquet(DATA_DIR / "transcripts_filtered.parquet", index=False)
print(f"Saved {len(transcripts_filtered):,} filtered transcripts.")
print("\nSummary:")
print(f"  Total raw transcripts:    {len(transcripts):,}")
print(f"  After gene filter:        {len(transcripts_genes):,}")
print(f"  After Q >= 20 filter:     {len(transcripts_filtered):,}")
print(f"  Unique genes:             {transcripts_filtered['feature_name'].nunique()}")
print(f"  10x cells:                {len(cells):,}")